In [ ]:
import pandas as pd
import re
import json

In [ ]:
df = pd.read_excel('../data/Depeche Mode Songs.xlsx')
df.head()

In [ ]:
# Drop the Compilation? column and the unnamed empty column E
df = df.drop(columns=['Compilation?', 'Unnamed: 5'], errors='ignore')

# Rename columns to clean JSON keys
df = df.rename(columns={'Version Type': 'Version_Type'})

df.head()

In [ ]:
def parse_writers(value):
    if pd.isna(value):
        return []
    writers = []
    for entry in str(value).split(';'):
        entry = entry.strip()
        match = re.match(r'^(.+?)\s*\(age\s*(.+?)\)$', entry)
        if match:
            writers.append({
                'name': match.group(1).strip(),
                'age': match.group(2).strip()
            })
        else:
            # Writer listed with no age (e.g. Andrew Phillpott)
            writers.append({'name': entry, 'age': None})
    return writers

df['Writers'] = df['Writers'].apply(parse_writers)
df[['Song', 'Writers']].head(10)

In [ ]:
string_cols = ['Song', 'Album', 'Version_Type', 'Notes']
for col in string_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

df.head()

In [ ]:
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Year'] = df['Year'].apply(lambda x: int(x) if pd.notna(x) else None)

df['Year'].head(20)

In [ ]:
df.to_json(
    '../data/depeche-songs.json',
    orient='records',
    force_ascii=False,
    indent=2
)
print("JSON exported successfully!")

In [ ]:
with open('../data/depeche-songs.json', 'r') as f:
    print(f.read(500))